In [ ]:
!pip install mlflow optuna lightgbm imbalanced-learn
!pip install dagshub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838

In [ ]:
#!aws configure
import dagshub
dagshub.init(repo_owner="rohitbedse",repo_name="yt-comment-sentiment-analysis",mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=8cdd4ac0-ae2f-4c9e-9b86-37f4bd52a00b&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=517e044f6e7e7f06c5f2cee927699615b3b74e2cd78be7bb6bee24a695ae46cf




Accessing as rohitbedse

In [3]:
import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow")

In [4]:
# Set or create an experiment
mlflow.set_experiment("Exp 5 - ML Algos with HP Tuning")

<Experiment: artifact_location='mlflow-artifacts:/0ffe6a81b68f4ec3b92bc9720802112d', creation_time=1774370761776, experiment_id='4', last_update_time=1774370761776, lifecycle_stage='active', name='Exp 5 - ML Algos with HP Tuning', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [8]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, classification_report, f1_score
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
import mlflow
import mlflow.sklearn
import optuna
import numpy as np

In [11]:
df = pd.read_csv('/content/reddit_preprocessing.csv').dropna()
df.shape

(36662, 2)

In [12]:
# -----------------------------
# STEP 1: Label Mapping
# -----------------------------
df = df.dropna(subset=['category'])
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# -----------------------------
# STEP 2: Show ORIGINAL label distribution
# -----------------------------
print("🔥 ORIGINAL LABELS:")
print(np.unique(df['category'], return_counts=True))

# -----------------------------
# STEP 3: TRAIN-TEST SPLIT (NO LEAKAGE)
# -----------------------------
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_comment'],
    df['category'],
    test_size=0.2,
    random_state=42,
    stratify=df['category']
)

# -----------------------------
# STEP 4: TF-IDF (FIT ONLY ON TRAIN)
# -----------------------------
vectorizer = TfidfVectorizer(ngram_range=(1,3), max_features=5000)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

# -----------------------------
# STEP 5: Show TRAIN distribution
# -----------------------------
print("\n📊 TRAIN LABELS:")
print(np.unique(y_train, return_counts=True))

# -----------------------------
# STEP 6: COMPUTE CLASS WEIGHTS
# -----------------------------
classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weight_dict = dict(zip(classes, class_weights))

print("\n⚖️ CLASS WEIGHTS:")
print(class_weight_dict)

# -----------------------------
# STEP 7: MLflow logging
# -----------------------------
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        mlflow.set_tag("mlflow.runName", f"{model_name}_CLASS_WEIGHT_TFIDF")
        mlflow.set_tag("experiment", "No_Leakage_Pipeline")

        mlflow.log_param("algo_name", model_name)
        mlflow.log_param("resampling", "None")
        mlflow.log_param("class_weight", "balanced")
        mlflow.log_param("vectorizer", "TF-IDF (1,3)")

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='macro')

        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_macro", f1)

        report = classification_report(y_test, y_pred, output_dict=True)

        for label, metrics in report.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        mlflow.sklearn.log_model(model, f"{model_name}_model")

# -----------------------------
# STEP 8: OPTUNA OBJECTIVE
# -----------------------------
def objective_lightgbm(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "random_state": 42
    }

    model = LGBMClassifier(
        **params,
        class_weight='balanced'   # 🔥 key change
    )

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train,
        test_size=0.2,
        random_state=42,
        stratify=y_train
    )

    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_val)

    return f1_score(y_val, y_pred, average='macro')

# -----------------------------
# STEP 9: RUN OPTUNA
# -----------------------------
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_lightgbm, n_trials=30)

    print("\n🏆 BEST PARAMS:", study.best_params)

    best_model = LGBMClassifier(
        **study.best_params,
        class_weight='balanced',
        random_state=42
    )

    log_mlflow("LightGBM", best_model, X_train, X_test, y_train, y_test)

# -----------------------------
# RUN
# -----------------------------
run_optuna_experiment()

🔥 ORIGINAL LABELS:
(array([0, 1, 2]), array([12644, 15770,  8248]))


[I 2026-04-09 07:51:39,920] A new study created in memory with name: no-name-ed226d72-f32f-43f8-aacb-8682ca52d728



📊 TRAIN LABELS:
(array([0, 1, 2]), array([10115, 12616,  6598]))

⚖️ CLASS WEIGHTS:
{np.int64(0): np.float64(0.9665183720547043), np.int64(1): np.float64(0.7749154512787995), np.int64(2): np.float64(1.481711629786804)}
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.529971 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 102400
[LightGBM] [Info] Number of data points in the train set: 23463, number of used features: 3555
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-04-09 07:51:55,476] Trial 0 finished with value: 0.5736166082623945 and parameters: {'n_estimators': 181, 'learning_rate': 0.0004777547600916534, 'max_depth': 9}. Best is trial 0 with value: 0.5736166082623945.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.740403 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 102400
[LightGBM] [Info] Number of data points in the train set: 23463, number of used features: 3555
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612


...

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.476484 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 102400
[LightGBM] [Info] Number of data points in the train set: 23463, number of used features: 3555
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-04-09 07:59:00,969] Trial 29 finished with value: 0.6334324658112166 and parameters: {'n_estimators': 193, 'learning_rate': 0.005518386823333425, 'max_depth': 9}. Best is trial 12 with value: 0.8367934994522708.



🏆 BEST PARAMS: {'n_estimators': 290, 'learning_rate': 0.09788879487759714, 'max_depth': 9}
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.776418 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 130547
[LightGBM] [Info] Number of data points in the train set: 29329, number of used features: 4361
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Wa

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/04/09 07:59:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 08:00:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LightGBM_CLASS_WEIGHT_TFIDF at: https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow/#/experiments/4/runs/c6779469edbd4501b4f0343f782c3a53
🧪 View experiment at: https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow/#/experiments/4


In [14]:
# ✅ Best params (converted properly)
best_params = {
    'n_estimators': 290,
    'learning_rate': 0.0978,
    'max_depth': 9,
    #'bagging_fraction': 0.68,      # subsample equivalent
    #'feature_fraction': 0.78,      # colsample_bytree equivalent
    'random_state': 42
}

# ⚠️ Important: enable bagging
model = LGBMClassifier(**best_params)

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluation
print("Test Macro F1:", f1_score(y_test, y_pred, average='macro'))
print("Test Weighted F1:", f1_score(y_test, y_pred, average='weighted'))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.981344 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 130547
[LightGBM] [Info] Number of data points in the train set: 29329, number of used features: 4361
[LightGBM] [Info] Start training from score -1.064557
[LightGBM] [Info] Start training from score -0.843611
[LightGBM] [Info] Start training from score -1.491810
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Test Macro F1: 0.8219717442373468
Test Weighted F1: 0.8371290364825626


In [15]:
import numpy as np

print("Unique predictions:", np.unique(y_pred))
print("Unique true labels:", np.unique(y_test))

Unique predictions: [0 1 2]
Unique true labels: [0 1 2]
